In [1]:
import os
import numpy as np
import pandas as pd
import cv2
import librosa
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix



import librosa
from moviepy.editor import VideoFileClip
from keras.applications import ResNet50
from keras.models import Model
from keras.layers import Conv2D, LSTM, Dense, Flatten, Concatenate, Input, Dropout
from keras.optimizers import Adam
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt
import dlib


import os
import numpy as np
import cv2
import librosa
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from tensorflow.keras.applications import ResNet50
from tensorflow.keras import layers, models
import tensorflow as tf


In [2]:

# Load ResNet model for feature extraction
def create_visual_model():
    """Create a visual feature extraction model using ResNet50."""
    base_model = ResNet50(weights='imagenet', include_top=False, pooling='avg')
    input_layer = layers.Input(shape=(224, 224, 3))
    x = base_model(input_layer)
    return models.Model(inputs=input_layer, outputs=x)

def create_audio_model(input_shape):
    """Create an audio feature extraction model."""
    input_layer = layers.Input(shape=input_shape)
    x = layers.Conv1D(64, kernel_size=3, activation='relu')(input_layer)
    x = layers.MaxPooling1D(pool_size=2)(x)
    x = layers.Conv1D(128, kernel_size=3, activation='relu')(x)
    x = layers.MaxPooling1D(pool_size=2)(x)
    x = layers.Flatten()(x)
    return models.Model(inputs=input_layer, outputs=x)

# Function to extract audio from a video
def extract_audio(video_path):
    clip = VideoFileClip(video_path)
    audio = clip.audio
    audio_path = video_path.replace(".mp4", ".wav")
    audio.write_audiofile(audio_path)
    return audio_path

def extract_visual_features(frames, model):
    """Extract visual features from video frames using ResNet model."""
    features = []
    for frame in frames:
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = cv2.resize(frame, (224, 224))
        frame = np.expand_dims(frame, axis=0) / 255.0  # Normalize
        feature = model.predict(frame)
        features.append(feature)
    return np.array(features).squeeze()

def extract_audio_features(audio_path, sequence_length=100):
    """Extract and pad/truncate MFCC features to match a fixed sequence length."""
    try:
       
        y, sr = librosa.load(audio_path, sr=None)
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
        
        if mfccs.shape[1] < sequence_length:
            mfccs = np.pad(mfccs, ((0, 0), (0, sequence_length - mfccs.shape[1])), mode='constant')
        else:
            mfccs = mfccs[:, :sequence_length]
        
        return mfccs.T
    except Exception as e:
        print(f"Error extracting audio features from {audio_path}: {e}")
        return None

import cv2
import numpy as np

def sample_frames(video_path, n_frames=5):
    """Sample frames from the video, with zero padding for videos with fewer frames."""
    cap = cv2.VideoCapture(video_path)
    extract_audio(video_path)
    if not cap.isOpened():
        print(f"Error: Cannot open video {video_path}")
        return np.zeros((n_frames, 224, 224, 3), dtype=np.uint8)  # Return zero-padded frames
    
    frames = []
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    # Determine the sampling interval based on frame count
    sample_interval = max(1, frame_count // n_frames)

    for i in range(n_frames):
        # Calculate the frame index to capture
        frame_index = min(i * sample_interval, frame_count - 1)
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_index)
        ret, frame = cap.read()
        if ret:
            frame = cv2.resize(frame, (224, 224))  # Resize to (224, 224)
            frames.append(frame)
        else:
            frames.append(np.zeros((224, 224, 3), dtype=np.uint8))  # Append zero frame if capture fails
    
    cap.release()
    
    # Convert frames to numpy array and pad with zeros if fewer than required frames
    frames = np.array(frames)
    if len(frames) < n_frames:
        padding = np.zeros((n_frames - len(frames), 224, 224, 3), dtype=np.uint8)
        frames = np.concatenate((frames, padding), axis=0)
    
    return frames


def preprocess_video_data(video_dir, metadata, visual_model):
    """Preprocess video data to extract audio and visual features."""
    video_features_list, audio_features_list = [], []
    labels = []
    
    for index, row in metadata.iterrows():
        video_id = row['VideoID']
        video_path = os.path.join(video_dir, f'{video_id}')
        
        video_frames = sample_frames(video_path, n_frames=5)
        print(video_frames)
        
        if len(video_frames) == 0:
            print(f"Warning: No frames for video {video_id}. Skipping.")
            continue
        
        video_features = extract_visual_features(video_frames, visual_model)
        if video_features.size == 0:
            print(f"Warning: No visual features for video {video_id}. Skipping.")
            continue
        
        audio_path = os.path.join(video_dir, f'{video_id}.wav')
        audio_features = extract_audio_features(audio_path)
        
        if audio_features is None or audio_features.size == 0:
            print(f"Warning: No audio features for video {video_id}. Skipping.")
            continue
        
        video_features_list.append(video_features)
        audio_features_list.append(audio_features)
        labels.append(row['HowFunny'])
    
    return np.array(video_features_list), np.array(audio_features_list), np.array(labels)

def train_and_evaluate_model(video_features, audio_features, labels):
    """Train and evaluate a deep learning model using intermediate fusion."""
    video_features_flat = video_features.reshape(video_features.shape[0], -1)
    
    combined_features = np.concatenate((video_features_flat, audio_features), axis=1)

    X_train, X_test, y_train, y_test = train_test_split(combined_features, labels, test_size=0.2, random_state=42)

    video_input = layers.Input(shape=(video_features_flat.shape[1],))
    audio_input = layers.Input(shape=(audio_features.shape[1],))

    video_branch = layers.Dense(128, activation='relu')(video_input)
    video_branch = layers.Dense(64, activation='relu')(video_branch)

    audio_branch = layers.Dense(128, activation='relu')(audio_input)
    audio_branch = layers.Dense(64, activation='relu')(audio_branch)

    merged = layers.concatenate([video_branch, audio_branch])
    merged = layers.Dense(128, activation='relu')(merged)
    output = layers.Dense(4, activation='softmax')(merged)

    model = models.Model(inputs=[video_input, audio_input], outputs=output)
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    model.fit([X_train[:, :video_features_flat.shape[1]], X_train[:, video_features_flat.shape[1]:]], y_train,
              epochs=10, batch_size=32, validation_split=0.2)

    y_pred = model.predict([X_test[:, :video_features_flat.shape[1]], X_test[:, video_features_flat.shape[1]:]])
    y_pred_classes = np.argmax(y_pred, axis=1)
    
    accuracy = accuracy_score(y_test, y_pred_classes)
    f1 = f1_score(y_test, y_pred_classes, average='weighted')
    conf_matrix = confusion_matrix(y_test, y_pred_classes)
    
    print(f'Accuracy: {accuracy:.2f}')
    print(f'F1 Score: {f1:.2f}')
    print(f'Confusion Matrix:\n{conf_matrix}')
    
    return model

In [3]:
# Ensure that the video directory and metadata file path are correctly set
video_dir = '../sdata'
metadata_file = "/Users/jeremiah/video-p-reaction-joketype-modular/Cursor_october/dataa/excel.xlsx"
metadata = pd.read_excel(metadata_file) 

In [4]:
# Initialize models
visual_model = create_visual_model()

In [5]:
# Example usage
sequence_length = 100  # Define a sequence length that works for your audio data
audio_model = create_audio_model((13, sequence_length))  # 13 MFCC features with a sequence length of 100

In [6]:
# Preprocess video data and metadata
video_features, audio_features, labels = preprocess_video_data(video_dir, metadata,visual_model)

OpenCV: Couldn't read video stream from file "../sdata/NFCK3A.joke1.rep1.take1.ThatsNotAHat.mp4"


OSError: MoviePy error: the file ../sdata/NFCK3A.joke1.rep1.take1.ThatsNotAHat.mp4 could not be found!
Please check that you entered the correct path.

In [ ]:
# Train the model
model = train_and_evaluate_model(video_features, audio_features, labels)

In [ ]:
# main('../data/excel.xlsx', '/Users/jeremiah/video-p-reaction-joketype-modular/Cursor_october/sdata',num_frames)